# NB_02 — Bronze SQL Profiling & Validation

**Project:** Nexa Commerce  
**Lakehouse:** Nexa_Lakehouse  
**Layer:** Bronze  
**Purpose:** Profile and investigate Bronze data using SQL before applying any transformations.

---

## Objective

This notebook uses SQL to understand the structure, quality, and relationships
of the source data ingested into the Bronze layer.

The goal is not to clean the data at this stage.

Instead, we will use SQL to:

- Understand the grain of each source table
- Profile key fields and data completeness
- Investigate duplicates and inconsistent identifiers
- Examine relationships between headers and lines
- Detect intentional source-quality issues
- Validate important business rules
- Document findings that will drive the Silver transformations

### Principle

**Profile first → Understand the problem → Transform later**

The Bronze layer should remain unchanged during this investigation.

## 1. Source Volume

### Business Questions

Before investigating data quality, we need to understand the size and coverage
of the Bronze data.

We will answer:

1. How many rows are present in each Bronze table?
2. How many orders exist in each sales channel?
3. How many order lines exist in each sales channel?
4. What is the date range covered by each major transactional source?
5. Which sources contain the largest volumes of data?

The results will establish a baseline for the subsequent profiling and validation
work.

### 1.1 How many rows are present in each Bronze table?

We need to establish the row-count baseline for all Bronze tables before
investigating data quality or relationships.

In [ ]:
SELECT "bronze_amazon_order_lines" AS table_name , COUNT (*) AS row_count
FROM bronze_amazon_order_lines

UNION ALL

SELECT "bronze_amazon_orders" AS table_name , COUNT (*) AS row_count
FROM bronze_amazon_orders

UNION ALL

SELECT "bronze_campaigns" AS table_name , COUNT (*) AS row_count
FROM bronze_campaigns

UNION ALL

FROM bronze_channeladvisor_order_lines

UNION ALL

FROM bronze_channeladvisor_order_lines

UNION ALL

SELECT "bronze_channeladvisor_orders" AS table_name , COUNT (*) AS row_count
FROM bronze_channeladvisor_orders


UNION ALL

SELECT "bronze_google_ads_campaign_daily" AS table_name , COUNT (*) AS row_count
FROM bronze_google_ads_campaign_daily


UNION ALL

SELECT "bronze_inventory_daily" AS table_name , COUNT (*) AS row_count
FROM bronze_inventory_daily

UNION ALL

SELECT "bronze_locations" AS table_name , COUNT (*) AS row_count
FROM bronze_locations

UNION ALL

SELECT "bronze_product_master" AS table_name , COUNT (*) AS row_count
FROM bronze_product_master

UNION ALL

SELECT "bronze_purchase_order_lines" AS table_name , COUNT (*) AS row_count
FROM bronze_purchase_order_lines

UNION ALL

SELECT "bronze_purchase_orders" AS table_name , COUNT (*) AS row_count
FROM bronze_purchase_orders

UNION ALL

SELECT "bronze_refund_events" AS table_name , COUNT (*) AS row_count
FROM bronze_refund_events

UNION ALL

SELECT "bronze_shopify_order_lines" AS table_name , COUNT (*) AS row_count
FROM bronze_shopify_order_lines

UNION ALL

SELECT "bronze_shopify_orders" AS table_name , COUNT (*) AS row_count
FROM bronze_shopify_orders

UNION ALL

SELECT "bronze_suppliers" AS table_name , COUNT (*) AS row_count
FROM bronze_suppliers

UNION ALL

SELECT "bronze_walmart_connect_weekly" AS table_name , COUNT (*) AS row_count
FROM bronze_walmart_connect_weekly

UNION ALL

SELECT "bronze_walmart_order_lines" AS table_name , COUNT (*) AS row_count
FROM bronze_walmart_order_lines

UNION ALL

SELECT "bronze_walmart_orders" AS table_name , COUNT (*) AS row_count
FROM bronze_walmart_orders

### 1.2 How is order volume distributed across sales channels?

We need to understand how the total order volume is distributed across the
different sales channels.

For each channel, we will calculate:

- Total number of orders
- Percentage of total orders

This provides a business-level view of channel contribution and establishes
a baseline for later cross-channel analysis.

In [22]:
WITH channel_orders AS

(

SELECT "Amazon" As channel,
COUNT (*) As order_count
FROM bronze_amazon_orders

UNION ALL

SELECT "Shopify" As channel,
COUNT (*) As order_count
FROM bronze_shopify_orders

UNION ALL

SELECT "Walmart" As channel,
COUNT (*) As order_count
FROM bronze_Walmart_orders

UNION ALL

SELECT "ChannelAdvisor" As channel,
COUNT (*) order_count
FROM bronze_channeladvisor_orders

),

channel_total AS
(
SELECT 
channel,
order_count,
SUM (order_count) OVER () AS total_orders
FROM channel_orders
)

SELECT 
channel,
order_count,
total_orders,
Order_count * 100 / total_orders  As percent_of_total
FROM channel_total;




StatementMeta(, 5b2827db-5680-40e0-8deb-78551827ff7b, 23, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 4 fields>

### 1.3 How is order-line volume distributed across sales channels?

Orders and order lines represent different levels of detail.

We will analyze the order-line grain separately to understand how many individual
product lines are generated by each sales channel.

For each channel, we will calculate:

- Total number of order lines
- Percentage of total order lines

This provides a baseline for understanding transaction detail across channels
before investigating order-to-line relationships and data quality.

In [28]:
WITH channel_oder_lines AS
(

SELECT "Amazon" as channel_name,
COUNT (*) AS order_lines_count
FROM bronze_amazon_order_lines

UNION ALL

SELECT "Shopify" as channel_name,
COUNT (*) AS order_lines_count
FROM bronze_shopify_order_lines

UNION ALL

SELECT "Walmart" as channel_name,
COUNT (*) AS order_lines_count
FROM bronze_walmart_order_lines


UNION ALL

SELECT "ChannelAdvisor" as channel_name,
COUNT (*) AS order_lines_count
FROM bronze_channeladvisor_order_lines

),

total_order_lines
(

SELECT
channel_name,
order_lines_count,
SUM (order_lines_count) OVER() AS total_order_lines
FROM channel_oder_lines
)

SELECT
channel_name,
order_lines_count,
total_order_lines,
order_lines_count *100 / total_order_lines AS percent_of_total
FROM total_order_lines



StatementMeta(, 5b2827db-5680-40e0-8deb-78551827ff7b, 29, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 4 fields>

## 2 — Data Quality Investigation

The Bronze layer contains source data from multiple systems, each with its own
identifiers, naming conventions, date logic, and data limitations.

The purpose of this profiling phase is to investigate the known data-quality and
integration problems before applying transformations.

We will use SQL to:

- Detect inconsistent identifiers
- Investigate duplicate and echo records
- Identify inconsistent channel naming
- Identify currency differences
- Investigate incomplete refund information
- Examine inventory coverage gaps
- Understand limitations in advertising attribution

The Bronze data will remain unchanged.

**Goal:** Understand the actual problems in the source data before deciding how
each problem should be handled in the Silver and Warehouse layers.

### 2.1 — Inconsistent Order ID Formats

Identify how order IDs are formatted across Amazon, ChannelAdvisor, Walmart, and Shopify.

Determine whether the source order IDs can be used directly across channels or require source-specific handling.

In [52]:
WITH orderid_format AS
            (
            ( SELECT "Amazon" AS channel_name,
                source_order_id AS order_id
                FROM bronze_amazon_orders
                LIMIT 3)
            
                UNION ALL

            ( SELECT "Channel_Advisor" AS channel_name,
                source_order_id AS Order_id
                FROM bronze_channeladvisor_orders
                LIMIT 3)
                
                UNION ALL

              ( SELECT "Walmart" AS channel_name,
                source_order_id AS order_id
                FROM bronze_walmart_orders
                LIMIT 3)

                UNION ALL

              (  SELECT "Shopify" AS channel_name,
                source_order_id AS order_id
                FROM bronze_shopify_orders
                LIMIT 3)
            )

            SELECT channel_name,
            order_id
            FROM orderid_format
            

StatementMeta(, b4b26080-ee01-4b47-afc3-8c5bd80b732c, 52, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 2 fields>

### 2.2 — ChannelAdvisor Duplicate Orders

Identify orders in ChannelAdvisor that are already present in the direct sales
channel data.

Determine how many duplicate orders exist and which direct channel they belong to.

In [75]:
SELECT
channel_name,
COUNT(source_order_id) AS total_orders
FROM bronze_channeladvisor_orders
GROUP BY channel_name

StatementMeta(, b4b26080-ee01-4b47-afc3-8c5bd80b732c, 75, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 2 fields>

In [74]:
SELECT
ca.source_order_id,
ca.channel_order_id,
ca.channel_name,
ca.order_total
FROM bronze_channeladvisor_orders ca
INNER JOIN bronze_amazon_orders ao
ON ca.channel_order_id = ao.source_order_id
INNER JOIN bronze_walmart_orders wo
ON ca.channel_order_id = wo.source_order_id
INNER JOIN bronze_shopify_orders so
ON ca.channel_order_id = so.source_order_id



StatementMeta(, b4b26080-ee01-4b47-afc3-8c5bd80b732c, 74, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 4 fields>

### Result

The ChannelAdvisor-to-Amazon join did not identify duplicate Amazon orders in
the current dataset.

The investigation confirmed that ChannelAdvisor `channel_order_id` is the
relevant identifier for matching an originating marketplace order.

### 2.3 — Duplicate Orders Within Sales Sources

Identify order IDs that appear more than once within each sales source.

Determine which sources contain duplicate orders and how many duplicate records exist.

In [3]:
SELECT
"Amazon" AS channe_name,
source_order_id,
COUNT (*) AS order_count
FROM bronze_amazon_orders
GROUP BY source_order_id
HAVING COUNT(*) > 1

UNION ALL

SELECT
"Walmart" AS channe_name,
source_order_id,
COUNT (*) AS order_count
FROM bronze_walmart_orders
GROUP BY source_order_id
HAVING COUNT(*) > 1

UNION ALL

SELECT
"Channel Advisor" AS channe_name,
source_order_id,
COUNT (*) AS order_count
FROM bronze_channeladvisor_orders
GROUP BY source_order_id
HAVING COUNT(*) > 1

UNION ALL

SELECT
"Shopify" AS channe_name,
source_order_id,
COUNT (*) AS order_count
FROM bronze_shopify_orders
GROUP BY source_order_id
HAVING COUNT(*) > 1

StatementMeta(, 80a261fc-7f56-4f6b-8233-5f324329427c, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>

### 2.4 — Currency Differences

Identify whether the source data contains multiple currencies and determine which records require currency normalization in Silver.

In [14]:
SELECT
"Amazon" AS channel_name,
currency
FRoM bronze_amazon_orders
GROUP BY currency

UNION ALL

SELECT
"Walmart" AS channel_name,
 currency
FRoM bronze_walmart_orders
GROUP BY currency

UNION ALL

SELECT
"ChannelAdvisor" AS channel_name,
 currency
FRoM bronze_channeladvisor_orders
GROUP BY currency

UNION ALL

SELECT
"Shopify" AS channel_name,
currency
FRoM bronze_shopify_orders
GROUP BY currency

StatementMeta(, 480e96e6-e5ba-43c7-8c55-8cda7b4fe391, 19, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 2 fields>

In [29]:
-- DESCRIBE bronze_amazon_orders
-- DESCRIBE bronze_walmart_orders
-- DESCRIBE bronze_channeladvisor_orders
-- DESCRIBE bronze_shopify_orders
-- DESCRIBE bronze_refund_events
DESCRIBE bronze_inventory_daily

StatementMeta(, 480e96e6-e5ba-43c7-8c55-8cda7b4fe391, 34, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 3 fields>

### 2.5 — Repeated Shopify Order Events

Identify Shopify orders that appear more than once and determine whether the duplicates represent repeated events for the same order.

In [20]:
SELECT
source_order_id,
COUNT (source_order_id) AS order_count
FROM bronze_shopify_orders
GROUP BY source_order_id
HAVING COUNT (source_order_id) > 1

StatementMeta(, 480e96e6-e5ba-43c7-8c55-8cda7b4fe391, 25, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

### 2.6 — Refund Records Without SKU Information

Check refund records that do not contain a SKU and determine how they can be linked back to the original order or order line.

In [1]:
SELECT
refund_id,
sku
FROM bronze_refund_events
WHERE sku IS NULL;

StatementMeta(, e41ec613-d414-4887-a2b5-1edb0ce89d91, 2, Finished, Available, Finished, False)

<Spark SQL result set with 214 rows and 2 fields>

In [2]:
--By source channel how much orders missing sku in refund_events table
SELECT 
source_channel,
COUNT(refund_id) AS refund_orders
FROM bronze_refund_events
WHERE sku IS NULL
GROUP BY source_channel

StatementMeta(, e41ec613-d414-4887-a2b5-1edb0ce89d91, 3, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

In [3]:
--Amazon by order_id available refund transactions detail 
SELECT
    refund_id, 
    source_order_id,
    source_channel,
    sku,
    quantity_refunded,
    refund_amount,
    currency
FROM bronze_refund_events
WHERE sku IS NULL
AND source_channel = "amazon"
LIMIT 10;

StatementMeta(, e41ec613-d414-4887-a2b5-1edb0ce89d91, 4, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 7 fields>

In [ ]:
--Amazon order_lines detail as per specific order to match with refund available data.
SELECT
    source_order_id,
    sku,
    quantity,
    unit_price,
    line_discount,
    line_tax,
    line_total
FROM bronze_amazon_order_lines
WHERE source_order_id = "AM-0000000758"

In [4]:
--ChannelAdvisor by order_id available refund transactions detail
SELECT
    refund_id,
    source_order_id, 
    source_channel,
    sku,
    quantity_refunded,
    refund_amount,
    currency
FROM bronze_refund_events
WHERE sku IS NULL
AND source_channel = "channeladvisor"
LIMIT 10;

StatementMeta(, e41ec613-d414-4887-a2b5-1edb0ce89d91, 5, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 7 fields>

In [5]:
--ChannelAdvisor order_lines detail as per specific order to match with refund available data.
SELECT
    source_order_id,
    sku,
    quantity,
    unit_price,
    line_discount,
    line_tax,
    line_total
FROM bronze_channeladvisor_order_lines
WHERE source_order_id = "CA-00003790"

StatementMeta(, e41ec613-d414-4887-a2b5-1edb0ce89d91, 6, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 7 fields>

In [6]:
--Walmart by order_id available refund transactions detail
SELECT
    refund_id, 
    source_order_id,
    source_channel,
    sku,
    quantity_refunded,
    refund_amount,
    currency
FROM bronze_refund_events
WHERE sku IS NULL
AND source_channel = "walmart"
LIMIT 10;

StatementMeta(, e41ec613-d414-4887-a2b5-1edb0ce89d91, 7, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 7 fields>

In [7]:
--Walmart order_lines detail as per specific order to match with refund available data.
SELECT
    source_order_id,
    sku,
    quantity,
    unit_price,
    line_discount,
    line_tax,
    line_total
FROM bronze_walmart_order_lines
WHERE source_order_id = "WM000002525"

StatementMeta(, e41ec613-d414-4887-a2b5-1edb0ce89d91, 8, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 7 fields>

### 2.7 — Inventory Date Coverage

Check whether each inventory location has a complete daily inventory record for the reporting period and identify any missing dates.

In [72]:
SELECT
    e.location_id,
    e.expected_date,
    i.snapshot_date
FROM (
    SELECT
    location_id,
    EXPLODE (SEQUENCE(
        MIN(TO_DATE(snapshot_date)),
        MAX(TO_DATE(snapshot_date)),
        INTERVAL 1 DAY 
             )
        ) AS expected_date 
        FROM bronze_inventory_daily
        GROUP BY location_id
        ) e
        LEFT JOIN bronze_inventory_daily i
        ON e.location_id = i.location_id
        AND e.expected_date = TO_DATE(i.snapshot_date)
        WHERE i.snapshot_date IS NULL

StatementMeta(, 480e96e6-e5ba-43c7-8c55-8cda7b4fe391, 77, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>

### 2.8 — Orders and Order Lines

Check whether every order has at least one matching order line in each sales source.

In [ ]:
SELECT 'Amazon' AS channel, ao.source_order_id
FROM bronze_amazon_orders ao
LEFT ANTI JOIN bronze_amazon_order_lines ol
    ON ao.source_order_id = ol.source_order_id

UNION ALL

SELECT 'Shopify' AS channel, so.source_order_id
FROM bronze_shopify_orders so
LEFT ANTI JOIN bronze_shopify_order_lines ol
    ON so.source_order_id = ol.source_order_id

UNION ALL

SELECT 'Walmart' AS channel, wo.source_order_id
FROM bronze_walmart_orders wo
LEFT ANTI JOIN bronze_walmart_order_lines ol
    ON wo.source_order_id = ol.source_order_id

UNION ALL

SELECT 'ChannelAdvisor' AS channel, ca.source_order_id
FROM bronze_channeladvisor_orders ca
LEFT ANTI JOIN bronze_channeladvisor_order_lines ol
    ON ca.source_order_id = ol.source_order_id

### 2.9 — Sales and Product Master

Check whether every SKU in the sales order lines exists in the product master.

In [10]:
SELECT
ol.sku
FROM (
SELECT sku FROM bronze_amazon_order_lines

UNION ALL

SELECT sku FROM bronze_channeladvisor_order_lines

UNION ALL

SELECT sku FROM bronze_walmart_order_lines

UNION ALL

SELECT sku FROM bronze_shopify_order_lines
) ol

LEFT ANTI JOIN bronze_product_master pm
ON ol.sku = pm.sku


StatementMeta(, 13920b9b-076a-4c21-a11b-0d98c7536c05, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

### 2.10 — Purchase Orders and Product Master

Check whether every SKU in purchase-order lines exists in the product master.

In [11]:
SELECT po.sku
FROM bronze_purchase_order_lines po
LEFT ANTI JOIN bronze_product_master pm
ON po.sku = pm.sku

StatementMeta(, 13920b9b-076a-4c21-a11b-0d98c7536c05, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

### 2.11 — Inventory and Product Master

Check whether every SKU in inventory exists in the product master.

In [ ]:
SELECT id.sku
FROM bronze_inventory_daily id
LEFT ANTI JOIN bronze_product_master pm
    ON id.sku = pm.sku

### 2.12 — Inventory and Locations

Check whether every inventory record references a valid location.

In [16]:
SELECT id.location_id
FROM bronze_inventory_daily id
LEFT ANTI JOIN bronze_locations bl
ON id.location_id = bl.location_id

StatementMeta(, 13920b9b-076a-4c21-a11b-0d98c7536c05, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

### 2.13 — Advertising Campaigns and Campaign Master

Check whether every campaign in the advertising data exists in the campaign master.

In [30]:
SELECT cc.campaign_id
FROM 
(
    SELECT campaign_id FROM bronze_google_ads_campaign_daily

    UNION ALL

    SELECT campaign_id FROM bronze_walmart_connect_weekly

) cc

LEFT ANTI JOIN bronze_campaigns bc
ON cc.campaign_id = bc.campaign_id

StatementMeta(, 13920b9b-076a-4c21-a11b-0d98c7536c05, 31, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

### 2.14 — Purchase Orders and Suppliers

Check whether every supplier in purchase orders exists in the supplier master.

In [33]:
SELECT po.supplier_id
FROM bronze_purchase_orders po
LEFT ANTI JOIN bronze_suppliers bs
ON po.supplier_id = bs.supplier_id

StatementMeta(, 13920b9b-076a-4c21-a11b-0d98c7536c05, 34, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>